In [1]:
import numpy as np
import pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt

# Zone layout 
zones = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
zone_to_idx = {z: i for i, z in enumerate(zones)}
idx_to_zone = {i: z for z, i in zone_to_idx.items()}

# Neighbors aka adjacency
neighbors = {
    'A': ['B', 'C'],
    'B': ['A', 'D'],
    'C': ['A', 'E'],
    'D': ['B', 'F'],
    'E': ['C', 'G'],
    'F': ['D', 'G'],
    'G': ['E', 'F']
}

# Passenger availability per zone (prob. of finding passenger)
passenger_prob = {
    'A': 0.9,   # High demand
    'B': 0.6,
    'C': 0.5,
    'D': 0.2,   # Low demand
    'E': 0.7,
    'F': 0.4,
    'G': 0.8
}

# Movement success rate
move_success_prob = 0.7
stay_prob = 1 - move_success_prob

print(f"Zones: {zones}")
print(f"Neighbors: {neighbors}")
print(f"Passenger probs: {passenger_prob}")

Zones: ['A', 'B', 'C', 'D', 'E', 'F', 'G']
Neighbors: {'A': ['B', 'C'], 'B': ['A', 'D'], 'C': ['A', 'E'], 'D': ['B', 'F'], 'E': ['C', 'G'], 'F': ['D', 'G'], 'G': ['E', 'F']}
Passenger probs: {'A': 0.9, 'B': 0.6, 'C': 0.5, 'D': 0.2, 'E': 0.7, 'F': 0.4, 'G': 0.8}


In [2]:
# State = (zone, passenger_status)
# passenger_status:
# 0 = no passenger, 1 = has passenger

states = [(z, p) for z in zones for p in [0, 1]]
state_to_idx = {s: i for i, s in enumerate(states)}
idx_to_state = {i: s for s, i in state_to_idx.items()}

# Actions: stay or move to any neighboring zone
def get_actions(zone):
    return ['stay'] + neighbors[zone]

print(f"Total states: {len(states)}")
print(f"Example state (A, empty): {state_to_idx[('A', 0)]}")
print(f"Example actions in A: {get_actions('A')}")

Total states: 14
Example state (A, empty): 0
Example actions in A: ['stay', 'B', 'C']


In [3]:
# Build transition and reward model for the MDP
def build_transition_matrix():
    """
    T[(state, action)] -> {next_state: probability}
    R[(state, action)] -> expected immediate reward
    """

    T = {}
    R = {}

    for state in states:

        zone, has_passenger = state

        for action in get_actions(zone):

            next_zone = zone if action == "stay" else action

            T[(state, action)] = {}
            expected_reward = 0

            # Two movement outcomes:
            # 0.7 -> intended move succeeds
            # 0.3 -> taxi remains in current zone
            movement_outcomes = [
                (move_success_prob, next_zone),
                (stay_prob, zone)
            ]

            for move_prob, result_zone in movement_outcomes:
                # Taxi already carrying a passenger
                if has_passenger == 1:

                    next_state = (result_zone, 1)

                    reward = -1      # only travel cost

                    T[(state, action)][next_state] = (
                        T[(state, action)].get(next_state, 0)
                        + move_prob
                    )

                    expected_reward += move_prob * reward

                # Taxi is empty and attempts to find a passenger     
                else:

                    pickup_probability = passenger_prob[result_zone]

                    # Passenger found
                    next_state_pickup = (result_zone, 1)
                    reward_pickup = -1 + 20

                    # No passenger found
                    next_state_empty = (result_zone, 0)
                    reward_empty = -1 - 5

                    prob_pickup = move_prob * pickup_probability
                    prob_empty = move_prob * (1 - pickup_probability)

                    T[(state, action)][next_state_pickup] = (
                        T[(state, action)].get(next_state_pickup, 0)
                        + prob_pickup
                    )

                    T[(state, action)][next_state_empty] = (
                        T[(state, action)].get(next_state_empty, 0)
                        + prob_empty
                    )

                    expected_reward += (
                        prob_pickup * reward_pickup
                        + prob_empty * reward_empty
                    )

            R[(state, action)] = expected_reward

    return T, R


T, R = build_transition_matrix()

print(f"Transition matrix size: {len(T)}")
print(f"Example transition for (A, empty), stay:\n{T[(('A', 0), 'stay')]}")
print(f"Expected reward: {R[(('A', 0), 'stay')]}")

Transition matrix size: 42
Example transition for (A, empty), stay:
{('A', 1): 0.9000000000000001, ('A', 0): 0.09999999999999998}
Expected reward: 16.500000000000004


In [4]:
def policy_evaluation(policy, T, R, gamma=0.9, theta=1e-4, max_iters=100):
    """
    Policy evaluation: compute V(s) for given policy
    """
    V = {s: 0.0 for s in states}
    
    for iteration in range(max_iters):
        delta = 0
        
        for state in states:
            action = policy[state]
            old_v = V[state]
            
            # V(s) = R(s,a) + gamma * sum_s' P(s'|s,a) * V(s')
            new_v = R[(state, action)]
            for next_state, prob in T[(state, action)].items():
                new_v += gamma * prob * V[next_state]
            
            V[state] = new_v
            delta = max(delta, abs(old_v - new_v))
        
        if delta < theta:
            print(f"Policy Evaluation converged in {iteration + 1} iterations")
            break
    
    return V

# Initialize random policy
init_policy = {}
for state in states:
    zone = state[0]
    actions = get_actions(zone)
    init_policy[state] = np.random.choice(actions)

print(f"Initial policy sample: {[(s, init_policy[s]) for s in states[:3]]}")

Initial policy sample: [(('A', 0), np.str_('B')), (('A', 1), np.str_('stay')), (('B', 0), np.str_('D'))]


In [5]:
def policy_improvement(policy, V, T, R, gamma=0.9):
    """
    Greedy policy update based on value function
    """
    policy_stable = True
    new_policy = {}
    
    for state in states:
        zone = state[0]
        actions = get_actions(zone)
        
        q_values = {}
        for action in actions:
            q = R[(state, action)]
            for next_state, prob in T[(state, action)].items():
                q += gamma * prob * V[next_state]
            q_values[action] = q
        
        best_action = max(q_values, key=q_values.get)
        new_policy[state] = best_action
        
        # Compare against CURRENT policy, not initial
        if best_action != policy[state]: 
            policy_stable = False
    
    return new_policy, policy_stable

In [6]:
def policy_iteration(T, R, gamma=0.9, max_iterations=50):
    """
    Full policy iteration
    """
    policy = init_policy.copy()
    iteration = 0
    
    for iteration in range(max_iterations):
        print(f"\n--- Iteration {iteration + 1} ---")
        
        # Evaluation
        V = policy_evaluation(policy, T, R, gamma)

        policy, stable = policy_improvement(policy, V, T, R, gamma)
        
        if stable:
            print(f"Policy converged in {iteration + 1} iterations!")
            break
    
    return policy, V

policy_opt, V_opt = policy_iteration(T, R, gamma=0.9)


--- Iteration 1 ---
Policy Evaluation converged in 89 iterations

--- Iteration 2 ---
Policy Evaluation converged in 89 iterations

--- Iteration 3 ---
Policy Evaluation converged in 89 iterations
Policy converged in 3 iterations!


In [7]:
# Optimal policy per zone
print("\n=== OPTIMAL POLICY ===")
zone_policy = {}
for zone in zones:
    for has_pass in [0, 1]:
        state = (zone, has_pass)
        passenger_str = "with passenger" if has_pass else "no passenger"
        print(f"Zone {zone} ({passenger_str}): {policy_opt[state]}")
    zone_policy[zone] = policy_opt[(zone, 0)]  # Policy when no passenger

print("\n=== VALUE FUNCTION ===")
for state in states:
    zone, has_pass = state
    passenger_str = "w/ pass" if has_pass else "no pass"
    print(f"V({zone}, {passenger_str}) = {V_opt[state]:.2f}")


=== OPTIMAL POLICY ===
Zone A (no passenger): stay
Zone A (with passenger): stay
Zone B (no passenger): A
Zone B (with passenger): A
Zone C (no passenger): A
Zone C (with passenger): stay
Zone D (no passenger): B
Zone D (with passenger): stay
Zone E (no passenger): G
Zone E (with passenger): stay
Zone F (no passenger): G
Zone F (with passenger): stay
Zone G (no passenger): stay
Zone G (with passenger): stay

=== VALUE FUNCTION ===
V(A, no pass) = 9.23
V(A, w/ pass) = -10.00
V(B, no pass) = 8.46
V(B, w/ pass) = -10.00
V(C, no pass) = 8.16
V(C, w/ pass) = -10.00
V(D, no pass) = 4.86
V(D, w/ pass) = -10.00
V(E, no pass) = 8.01
V(E, w/ pass) = -10.00
V(F, no pass) = 7.07
V(F, w/ pass) = -10.00
V(G, no pass) = 8.29
V(G, w/ pass) = -10.00


In [8]:
def value_iteration(T, R, gamma=0.9, theta=1e-4, max_iters=1000):
    """
    Value iteration (bellman equation)
    """
    V = {s: 0.0 for s in states}
    
    for iteration in range(max_iters):
        delta = 0
        
        for state in states:
            zone = state[0]
            actions = get_actions(zone)
            old_v = V[state]
            
            # V(s) = max_a [R(s,a) + gamma * sum_s' P(s'|s,a) * V(s')]
            best_value = float('-inf')
            for action in actions:
                q = R[(state, action)]
                for next_state, prob in T[(state, action)].items():
                    q += gamma * prob * V[next_state]
                best_value = max(best_value, q)
            
            V[state] = best_value
            delta = max(delta, abs(old_v - V[state]))
        
        if delta < theta:
            print(f"Value Iteration converged in {iteration + 1} iterations")
            break
    
    return V

gamma = 0.9

# Value iteration
V_vi = value_iteration(T, R, gamma=gamma)

# Extract policy from value iteration
policy_vi = {}
for state in states:
    zone = state[0]
    actions = get_actions(zone)
    q_values = {}
    for action in actions:
        q = R[(state, action)]
        for next_state, prob in T[(state, action)].items():
            q += gamma * prob * V_vi[next_state]
        q_values[action] = q
    policy_vi[state] = max(q_values, key=q_values.get)

print("\n=== COMPARISON: Policy Iteration vs Value Iteration ===")
print(f"Policies match: {policy_opt == policy_vi}")
print(f"Max V difference: {max(abs(V_opt[s] - V_vi[s]) for s in states):.6f}")

Value Iteration converged in 89 iterations

=== COMPARISON: Policy Iteration vs Value Iteration ===
Policies match: False
Max V difference: 0.000085


Policy Iteration:
  - Alternates between policy evaluation (compute V for current policy)
    and policy improvement (update policy greedily)
  - Converged in only a few policy improvement iterations
  - Each policy evaluation step is computationally more expensive

Value Iteration:
  - Combines policy evaluation and improvement into a single Bellman update
  - Updates the value function directly until convergence
  - Required more Bellman updates than Policy Iteration for this problem
  - Simpler per iteration but may require many more iterations

For Taxi Fleet:
  - High-demand zones (A, E, G) are preferred because their higher passenger availability leads to greater expected long-term rewards.
  - Low-demand zones (e.g., D) are visited only when they lead to better long-term returns
  - Empty taxis are encouraged to move toward high-demand zones to maximize expected future rewards
  - Both Policy Iteration and Value Iteration converged to the same optimal policy and value function for this MDP